## 2.Feature Engineering and Modelling

##### 1. Libraries Import

In [1]:
import os
import joblib 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)

In [2]:
import mlflow 
import mlflow.sklearn

# Setting up MLflow tracking URI and experiment name
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("Heart Disease Prediction")

print("MLFlow experiment set to 'Heart Disease Prediction' ")

MLFlow experiment set to 'Heart Disease Prediction' 


##### 2. Dataset Loading and Feature Preprocessing Pipeline

In [3]:
df = pd.read_csv("../data/heart_disease_cleaned.csv")

X = df.drop(columns=['target'])
y = df['target']

#Defining Feature subset based on UCI Heart Disease Dataset documentation
num_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat_features = ['cp', 'restecg', 'slope', 'ca', 'thal']
bin_features = ['sex', 'fbs', 'exang']

In [4]:
#Preprocessing Pipeline

preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ('bin', 'passthrough', bin_features)
    ]
)

# Stratified Train Test SPlit

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (237, 14)
Test set size: (60, 14)


##### 3. Model Training & MLFlow Logging

In [5]:
def train_evaluate_and_log(model_name, model, params):
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    with mlflow.start_run(run_name=model_name):
        #1. log Hyperparameters
        mlflow.log_params(params)

        # 2. Stratified 5-Fold Cross Validation
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_results = cross_validate(
            pipeline, X_train, y_train, cv=cv, 
            scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
        )

        # Logging Mean CV metrics
        cv_metrics = {
            "cv_mean_accuracy": cv_results['test_accuracy'].mean(),
            "cv_mean_precision": cv_results['test_precision'].mean(),
            "cv_mean_recall": cv_results['test_recall'].mean(),
            "cv_mean_f1": cv_results['test_f1'].mean(),
            "cv_mean_roc_auc": cv_results['test_roc_auc'].mean(),
        }
        mlflow.log_metrics(cv_metrics)

        #3. Fit model on Full Training & Evaluate on Test Set
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        test_metrics = {
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_precision": precision_score(y_test, y_pred),
            "test_recall": recall_score(y_test, y_pred),
            "test_f1": f1_score(y_test, y_pred),
            "test_roc_auc": roc_auc_score(y_test, y_proba)
        }
        mlflow.log_metrics(test_metrics)

        # 4. Generating and Logging Confusion Matrix
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_estimator(pipeline, X_test, y_test, ax=ax, cmap='Blues')
        plt.title(f"{model_name} - Confusion Matrix")
        cm_path = f"../visualizations/{model_name.lower().replace(' ', '_')}_confusion_matrix.png"
        plt.tight_layout()
        plt.savefig(cm_path, dpi=900)
        plt.close()
        mlflow.log_artifact(cm_path, artifact_path="confusion_matrix")

        # 5. Logging Pipeline Artifact to MLFlow
        mlflow.sklearn.log_model(pipeline, artifact_path="model")


        print(f"--- {model_name} Performance Metrics ---")
        for metric, value in test_metrics.items():
            print(f"{metric}: {value:.4f}")

        return pipeline

##### 4.1 Model 1 - Logistic Regression - Sample Run

In [6]:
lr_params = {
    "C": 1.0,
    "max_iter": 100,
    "solver": "lbfgs",
    "random_state": 42
}

lr_model = LogisticRegression(**lr_params)
lr_pipeline = train_evaluate_and_log("Logistic Regression", lr_model, lr_params)

2026/10/06 05:00:35 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



--- Logistic Regression Performance Metrics ---
test_accuracy: 0.8167
test_precision: 0.8696
test_recall: 0.7143
test_f1: 0.7843
test_roc_auc: 0.9375


##### 4.2 Model 2 - Random Forest Classifier - Sample Run

In [7]:
rf_params = {
    "n_estimators": 100,
    "max_depth": 5,
    "min_samples_split": 4,
    "random_state": 42
}

rf_model = RandomForestClassifier(**rf_params)
rf_pipeline = train_evaluate_and_log("Random Forest Classifier", rf_model, rf_params)

--- Random Forest Classifier Performance Metrics ---
test_accuracy: 0.8500
test_precision: 0.8800
test_recall: 0.7857
test_f1: 0.8302
test_roc_auc: 0.9498


c:\Users\testauto\Desktop\MLOps-Assignment-1\venv\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\testauto\Desktop\MLOps-Assignment-1\venv\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


##### 4.3 Hyperparameter Tuning and Model Selection

In [8]:
from sklearn.model_selection import ParameterGrid

# Comparing 12 configurations (6 per model) with identical stratified 5-fold CV.
# Selecting by mean validation ROC-AUC; using holdout test set only for the winner.
lr_configurations = list(ParameterGrid({"C": [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]}))
rf_configurations = list(ParameterGrid({
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8],
}))
candidates = (
    [("Logistic Regression", params) for params in lr_configurations]
    + [("Random Forest", params) for params in rf_configurations]
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tuning_results = []

for experiment_number, (model_name, params) in enumerate(candidates, start=1):
    if model_name == "Logistic Regression":
        estimator = LogisticRegression(max_iter=1000, random_state=42, **params)
    else:
        estimator = RandomForestClassifier(random_state=42, n_jobs=-1, **params)

    candidate_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator),
    ])
    config_label = ", ".join(f"{key}={value}" for key, value in params.items())

    with mlflow.start_run(run_name=f"Tuning {experiment_number:02d} - {model_name} ({config_label})"):
        mlflow.log_params({
            "model_type": model_name,
            "cv_folds": 5,
            "selection_metric": "roc_auc",
            **params,
        })
        cv_scores = cross_validate(
            candidate_pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
            error_score="raise",
        )
        cv_metrics = {}
        for metric_name in scoring:
            fold_scores = cv_scores[f"test_{metric_name}"]
            cv_metrics[f"cv_mean_{metric_name}"] = float(fold_scores.mean())
            cv_metrics[f"cv_std_{metric_name}"] = float(fold_scores.std(ddof=1))
        mlflow.log_metrics(cv_metrics)

    tuning_results.append({
        "model_name": model_name,
        "params": params,
        **cv_metrics,
    })

results_df = pd.DataFrame(tuning_results).sort_values(
    "cv_mean_roc_auc", ascending=False
).reset_index(drop=True)
display(results_df)


,model_name,params,cv_mean_accuracy,cv_std_accuracy,cv_mean_precision,cv_std_precision,cv_mean_recall,cv_std_recall,cv_mean_f1,cv_std_f1,cv_mean_roc_auc,cv_std_roc_auc
0,Logistic Regression,{'C': 0.5},0.834929,0.078101,0.853222,0.101453,0.787446,0.120061,0.813163,0.086015,0.903850,0.078962
1,Logistic Regression,{'C': 1.0},0.826507,0.083677,0.841984,0.103879,0.778355,0.123135,0.803625,0.091563,0.902057,0.078674
2,Logistic Regression,{'C': 0.1},0.822429,0.068998,0.853043,0.093367,0.751082,0.104883,0.794444,0.077778,0.900997,0.081702
3,Logistic Regression,{'C': 5.0},0.826507,0.087747,0.842324,0.112900,0.778788,0.116677,0.804837,0.092277,0.899529,0.078687
4,Logistic Regression,{'C': 10.0},0.826507,0.087747,0.842324,0.112900,0.778788,0.116677,0.804837,0.092277,0.899177,0.078145
5,Random Forest,"{'max_depth': 5, 'n_estimators': 200}",0.818528,0.057410,0.847121,0.082641,0.751082,0.131145,0.788351,0.073283,0.894112,0.066804
6,Random Forest,"{'max_depth': 5, 'n_estimators': 100}",0.814273,0.050869,0.855058,0.091039,0.732900,0.117199,0.781607,0.063165,0.892352,0.067877
7,Random Forest,"{'max_depth': 3, 'n_estimators': 200}",0.814273,0.064592,0.849318,0.103199,0.741991,0.122695,0.783985,0.075120,0.891110,0.070669
8,Random Forest,"{'max_depth': 8, 'n_estimators': 200}",0.814184,0.064958,0.829887,0.084593,0.760173,0.122494,0.787470,0.077021,0.887845,0.065080
9,Random Forest,"{'max_depth': 3, 'n_estimators': 100}",0.814273,0.064592,0.849318,0.103199,0.741991,0.122695,0.783985,0.075120,0.885712,0.076894


##### 4.4 RBF-Kernel Support Vector Machine


In [9]:
from sklearn.svm import SVC

svc_configurations = list(ParameterGrid({
    "C": [0.1, 1.0, 10.0],
    "gamma": ["scale", 0.01, 0.1],
}))
svc_results = []

for experiment_number, params in enumerate(svc_configurations, start=1):
    candidate_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", SVC(kernel="rbf", **params)),
    ])
    config_label = ", ".join(f"{key}={value}" for key, value in params.items())

    with mlflow.start_run(run_name=f"RBF SVM {experiment_number:02d} ({config_label})"):
        mlflow.log_params({
            "model_type": "RBF SVM",
            "cv_folds": 5,
            "selection_metric": "roc_auc",
            **params,
        })
        cv_scores = cross_validate(
            candidate_pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
            error_score="raise",
        )
        cv_metrics = {}
        for metric_name in scoring:
            fold_scores = cv_scores[f"test_{metric_name}"]
            cv_metrics[f"cv_mean_{metric_name}"] = float(fold_scores.mean())
            cv_metrics[f"cv_std_{metric_name}"] = float(fold_scores.std(ddof=1))
        mlflow.log_metrics(cv_metrics)

    svc_results.append({
        "model_name": "RBF SVM",
        "params": params,
        **cv_metrics,
    })

svc_results_df = pd.DataFrame(svc_results)
comparison_df = pd.concat(
    [results_df, svc_results_df], ignore_index=True
).sort_values("cv_mean_roc_auc", ascending=False).reset_index(drop=True)
display(comparison_df)

c:\Users\testauto\Desktop\MLOps-Assignment-1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\testauto\Desktop\MLOps-Assignment-1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


,model_name,params,cv_mean_accuracy,cv_std_accuracy,cv_mean_precision,cv_std_precision,cv_mean_recall,cv_std_recall,cv_mean_f1,cv_std_f1,cv_mean_roc_auc,cv_std_roc_auc
0,Logistic Regression,{'C': 0.5},0.834929,0.078101,0.853222,0.101453,0.787446,0.120061,0.813163,0.086015,0.903850,0.078962
1,RBF SVM,"{'C': 10.0, 'gamma': 0.01}",0.822518,0.080528,0.833807,0.100589,0.778355,0.123135,0.799725,0.089774,0.902220,0.069839
2,Logistic Regression,{'C': 1.0},0.826507,0.083677,0.841984,0.103879,0.778355,0.123135,0.803625,0.091563,0.902057,0.078674
3,Logistic Regression,{'C': 0.1},0.822429,0.068998,0.853043,0.093367,0.751082,0.104883,0.794444,0.077778,0.900997,0.081702
4,Logistic Regression,{'C': 5.0},0.826507,0.087747,0.842324,0.112900,0.778788,0.116677,0.804837,0.092277,0.899529,0.078687
5,Logistic Regression,{'C': 10.0},0.826507,0.087747,0.842324,0.112900,0.778788,0.116677,0.804837,0.092277,0.899177,0.078145
6,RBF SVM,"{'C': 1.0, 'gamma': 0.01}",0.822340,0.073962,0.872462,0.110618,0.732900,0.108026,0.790918,0.079942,0.896472,0.073670
7,Random Forest,"{'max_depth': 5, 'n_estimators': 200}",0.818528,0.057410,0.847121,0.082641,0.751082,0.131145,0.788351,0.073283,0.894112,0.066804
8,Random Forest,"{'max_depth': 5, 'n_estimators': 100}",0.814273,0.050869,0.855058,0.091039,0.732900,0.117199,0.781607,0.063165,0.892352,0.067877
9,Random Forest,"{'max_depth': 3, 'n_estimators': 200}",0.814273,0.064592,0.849318,0.103199,0.741991,0.122695,0.783985,0.075120,0.891110,0.070669


##### 4.5 Gradient Boosting Classifier

In [10]:
from sklearn.ensemble import GradientBoostingClassifier

gb_configurations = list(ParameterGrid({
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.03, 0.05, 0.1, 0.2],
    "max_depth": [1, 2],
}))
gb_results = []

for experiment_number, params in enumerate(gb_configurations, start=1):
    candidate_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(random_state=42, **params)),
    ])
    config_label = ", ".join(f"{key}={value}" for key, value in params.items())

    with mlflow.start_run(run_name=f"Gradient Boosting {experiment_number:02d} ({config_label})"):
        mlflow.log_params({
            "model_type": "Gradient Boosting",
            "cv_folds": 5,
            "selection_metric": "roc_auc",
            **params,
        })
        cv_scores = cross_validate(
            candidate_pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
            error_score="raise",
        )
        cv_metrics = {}
        for metric_name in scoring:
            fold_scores = cv_scores[f"test_{metric_name}"]
            cv_metrics[f"cv_mean_{metric_name}"] = float(fold_scores.mean())
            cv_metrics[f"cv_std_{metric_name}"] = float(fold_scores.std(ddof=1))
        mlflow.log_metrics(cv_metrics)

    gb_results.append({
        "model_name": "Gradient Boosting",
        "params": params,
        **cv_metrics,
    })

gb_results_df = pd.DataFrame(gb_results)
all_results_df = pd.concat(
    [results_df, svc_results_df, gb_results_df], ignore_index=True
).sort_values("cv_mean_roc_auc", ascending=False).reset_index(drop=True)
display(all_results_df)

,model_name,params,cv_mean_accuracy,cv_std_accuracy,cv_mean_precision,cv_std_precision,cv_mean_recall,cv_std_recall,cv_mean_f1,cv_std_f1,cv_mean_roc_auc,cv_std_roc_auc
0,Logistic Regression,{'C': 0.5},0.834929,0.078101,0.853222,0.101453,0.787446,0.120061,0.813163,0.086015,0.903850,0.078962
1,RBF SVM,"{'C': 10.0, 'gamma': 0.01}",0.822518,0.080528,0.833807,0.100589,0.778355,0.123135,0.799725,0.089774,0.902220,0.069839
2,Logistic Regression,{'C': 1.0},0.826507,0.083677,0.841984,0.103879,0.778355,0.123135,0.803625,0.091563,0.902057,0.078674
3,Logistic Regression,{'C': 0.1},0.822429,0.068998,0.853043,0.093367,0.751082,0.104883,0.794444,0.077778,0.900997,0.081702
4,Gradient Boosting,"{'learning_rate': 0.05, 'max_depth': 1, 'n_est...",0.801507,0.063701,0.820216,0.104405,0.750649,0.125377,0.774500,0.072891,0.900141,0.068188
5,Logistic Regression,{'C': 5.0},0.826507,0.087747,0.842324,0.112900,0.778788,0.116677,0.804837,0.092277,0.899529,0.078687
6,Logistic Regression,{'C': 10.0},0.826507,0.087747,0.842324,0.112900,0.778788,0.116677,0.804837,0.092277,0.899177,0.078145
7,Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 1, 'n_esti...",0.805674,0.053774,0.826670,0.091817,0.750649,0.137181,0.776778,0.068128,0.898686,0.064961
8,Gradient Boosting,"{'learning_rate': 0.2, 'max_depth': 1, 'n_esti...",0.809840,0.055223,0.828249,0.092216,0.759740,0.136750,0.782586,0.069742,0.898328,0.065727
9,Gradient Boosting,"{'learning_rate': 0.2, 'max_depth': 1, 'n_esti...",0.801507,0.072038,0.815981,0.115729,0.760173,0.109113,0.778934,0.070711,0.898295,0.070535


In [11]:
# Select the top cross-validation result, then evaluate it on the held-out test split.
best_result = all_results_df.iloc[0]
best_model_name = best_result["model_name"]
best_params = best_result["params"]

if best_model_name == "Logistic Regression":
    best_estimator = LogisticRegression(max_iter=1000, random_state=42, **best_params)
elif best_model_name == "Random Forest":
    best_estimator = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)
elif best_model_name == "RBF SVM":
    best_estimator = SVC(kernel="rbf", probability=True, random_state=42, **best_params)
else:
    best_estimator = GradientBoostingClassifier(random_state=42, **best_params)

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", best_estimator),
])
final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)
y_proba = final_pipeline.predict_proba(X_test)[:, 1]
final_test_metrics = {
    "test_accuracy": float(accuracy_score(y_test, y_pred)),
    "test_precision": float(precision_score(y_test, y_pred)),
    "test_recall": float(recall_score(y_test, y_pred)),
    "test_f1": float(f1_score(y_test, y_pred)),
    "test_roc_auc": float(roc_auc_score(y_test, y_proba)),
}

with mlflow.start_run(run_name=f"Final - {best_model_name}"):
    mlflow.log_params({
        "selected_model": best_model_name,
        "selection_metric": "cv_mean_roc_auc",
        "best_cv_mean_roc_auc": float(best_result["cv_mean_roc_auc"]),
        **best_params,
    })
    mlflow.log_metrics(final_test_metrics)
    mlflow.sklearn.log_model(final_pipeline, artifact_path="model")

print(f"Selected model: {best_model_name}")
print(f"Best CV ROC-AUC: {best_result['cv_mean_roc_auc']:.4f}")
print("Final holdout test metrics:")
for metric_name, value in final_test_metrics.items():
    print(f"{metric_name}: {value:.4f}")

Selected model: Logistic Regression
Best CV ROC-AUC: 0.9039
Final holdout test metrics:
test_accuracy: 0.8167
test_precision: 0.8696
test_recall: 0.7143
test_f1: 0.7843
test_roc_auc: 0.9364


c:\Users\testauto\Desktop\MLOps-Assignment-1\venv\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\testauto\Desktop\MLOps-Assignment-1\venv\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


##### 5. Export Final Production Model

In [12]:
# Save the selected model pipeline locally for the FastAPI service
model_export_path = "../model.pkl"
joblib.dump(final_pipeline, model_export_path)

print(f"Best model pipeline ({best_model_name}) exported successfully to '{model_export_path}'")

Best model pipeline (Logistic Regression) exported successfully to '../model.pkl'
